In [0]:
artists=spark.read.parquet("/Volumes/big-data-project/data/raw-data/artists.parquet",inferschema=True,header=True)
display(artists)

artist_uri,artist_name
spotify:artist:3TVXtAsR1Inumwj472S9r4,Drake
spotify:artist:1UwTwoC4T1i6vzwsQgIWB0,Feza
spotify:artist:1RyvyyTE3xzB2ZywiAwp0i,Future
spotify:artist:1zEHBw7xQf0drXZagRkciU,Molly Santana
spotify:artist:1URnnhqYAYcrqrcwql10ft,21 Savage
spotify:artist:2zMD4U9OQAR3xuq6cjer8p,Calvin Fallo
spotify:artist:0ysbLY7TUvj3AKV2W7ZCFY,LIMIT NALA
spotify:artist:2HPaUgqeutzr3jx5a9WyDV,PARTYNEXTDOOR
spotify:artist:5iEByMj7fW1lewiS4Ik91v,Trechyson Molly vx
spotify:artist:0fqv3yhjGZJ20OUjtIxj9H,DJ Promatic SA


Data Overview

In [0]:
print("="*90)
print("DATASET OVERVIEW")
print("="*90)

print(f"Total Records : {artists.count()}")
print(f"Total Columns : {len(artists.columns)}")

artists.printSchema()

DATASET OVERVIEW
Total Records : 63775
Total Columns : 2
root
 |-- artist_uri: string (nullable = true)
 |-- artist_name: string (nullable = true)



The dataset contains 63,775 unique artist records, representing the master list of artists available in the Spotify dataset.
It consists of 2 columns, making it a lightweight reference table that can be joined with other datasets such as songs, albums, and chart data using artist_uri.
Both columns are of String data type, which is appropriate since they store textual identifiers and artist names.
No numerical or temporal attributes are present; therefore, statistical analyses such as descriptive statistics and correlation analysis are not applicable for this dataset.
The dataset serves as a dimension (lookup) table in the overall data model and is primarily intended for metadata enrichment and relationship mapping across Spotify datasets.

Data Type Validation

In [0]:
print("="*90)
print("COLUMN DATA TYPES")
print("="*90)

for col_name, dtype in artists.dtypes:
    print(f"{col_name:<20} {dtype}")

COLUMN DATA TYPES
artist_uri           string
artist_name          string


Missing Value Analysis

In [0]:
from pyspark.sql.functions import *

In [0]:
print("="*90)
print("MISSING VALUE ANALYSIS")
print("="*90)

missing = artists.select([
    count(
        when(
            col(c).isNull() |
            (trim(col(c))=="") |
            (col(c)=="NULL") |
            (col(c)=="\\N"),
            c
        )
    ).alias(c)
    for c in artists.columns
])

missing.show()

MISSING VALUE ANALYSIS
+----------+-----------+
|artist_uri|artist_name|
+----------+-----------+
|         0|          2|
+----------+-----------+



The dataset exhibits excellent data quality, with only 2 missing values out of 63,775 records.
artist_uri has no missing values, ensuring that every artist has a valid Spotify identifier.
artist_name contains 2 missing values (approximately 0.0031% of the dataset), which is negligible and unlikely to impact downstream analysis.
Since artist_uri is available for all records, the missing artist names can be:
retained if only the identifier is required, or
removed/imputed during data preprocessing if artist names are essential for reporting or visualization.

Duplicate Record Analysis

In [0]:
print("="*90)
print("DUPLICATE RECORD ANALYSIS")
print("="*90)

total_records = artists.count()
unique_records = artists.dropDuplicates().count()
duplicate_records = total_records - unique_records

print(f"Total Records     : {total_records}")
print(f"Unique Records    : {unique_records}")
print(f"Duplicate Records : {duplicate_records}")

DUPLICATE RECORD ANALYSIS
Total Records     : 63775
Unique Records    : 63775
Duplicate Records : 0


The dataset contains 63,775 total records, all of which are unique.
No duplicate records were identified, indicating that each row represents a distinct artist entry.
The absence of duplicate records confirms that the dataset maintains high data integrity and is suitable for use as a master reference table.
Since no duplicate rows exist, no deduplication or additional preprocessing is required at the record level.

Column Level Analysis - Duplicate Artist URI Analysis

In [0]:
print("="*90)
print("ARTIST URI UNIQUENESS")
print("="*90)

duplicate_uri = artists.groupBy("artist_uri") \
    .count() \
    .filter(col("count") > 1)

print(f"Duplicate Artist URIs : {duplicate_uri.count()}")

duplicate_uri.show(20, truncate=False)

ARTIST URI UNIQUENESS
Duplicate Artist URIs : 0
+----------+-----+
|artist_uri|count|
+----------+-----+
+----------+-----+



The artist_uri column contains no duplicate values, confirming that every artist is associated with a unique Spotify identifier.
This validates artist_uri as the primary key for the dataset and makes it suitable for joining with other Spotify datasets such as albums, tracks, artwork, and chart data.
The uniqueness of the identifier ensures reliable relationships across datasets and prevents issues such as one-to-many ambiguities during data integration.
Combined with the absence of duplicate records, the dataset demonstrates strong referential integrity and can safely be used as a master lookup table.


Duplicate Artist Name Analysis


In [0]:
print("="*90)
print("ARTIST NAME DUPLICATE ANALYSIS")
print("="*90)

duplicate_names = (
    artists.groupBy("artist_name")
           .count()
           .filter(col("count") > 1)
           .orderBy(desc("count"))
)

print(f"Duplicate Artist Names : {duplicate_names.count()}")

# duplicate_names.show(20, truncate=False)
display(duplicate_names)

ARTIST NAME DUPLICATE ANALYSIS
Duplicate Artist Names : 1842


artist_name,count
Kali,8
Ian,8
Hiro,7
Danny,6
Adam,6
Maestro,6
Shenge Wasehlalankosi,6
TNT,5
Mario,5
Mjolisi,5


The dataset contains 1,842 artist names that appear more than once.
Unlike artist_uri, artist_name is not a unique identifier and duplicate names are expected in large music catalogs.
These duplicates may occur because:
Different artists share the same stage or display name.
Artists from different countries or genres use identical names.
Legacy or migrated artist profiles exist within the Spotify ecosystem.
Since every artist_uri is unique, these duplicate names do not represent duplicate records and therefore do not indicate a data quality issue.
For analytical tasks such as joins, aggregations, or dashboard development, artist_uri should always be used as the primary key, while artist_name should be treated as a descriptive attribute.

Unique Value Analysis

In [0]:
print("="*90)
print("UNIQUE VALUE ANALYSIS")
print("="*90)

for column in artists.columns:
    unique_count = artists.select(column).distinct().count()
    print(f"{column:<20} : {unique_count:,}")

UNIQUE VALUE ANALYSIS
artist_uri           : 63,775
artist_name          : 61,587


The artist_uri column contains 63,775 unique values, which matches the total number of records. This confirms that each artist is assigned a unique Spotify identifier.
The artist_name column contains 61,587 unique values, indicating that some artist names are shared by multiple Spotify artist profiles.
The difference between the total number of records and the number of unique artist names confirms that display names are not unique identifiers and should not be used as primary keys for joins or aggregations.
This result is consistent with the earlier duplicate analysis, where 1,842 artist names were found to occur more than once.